# 02 — Tracking pipeline

This notebook contains the core computer-vision pipeline used in the academic project **Player Movement Tracking using Computer Vision**.

It performs object detection and multi-object tracking, applies the project ID-management logic, classifies teams from HSV jersey colours, estimates pitch coordinates and exports both a tracking CSV and an annotated video.

> **Important:** this is a research prototype. The original logic is preserved; repository-specific changes are limited to documentation, removal of stored outputs and portable file paths.


## Expected local inputs

Before running the notebook:

- place the trained detector at `models/best.pt`;
- place a permitted test clip at `data/input/sample.mp4`;
- calibrate `TEAM_A_HSV` and `TEAM_B_HSV` with `01_hsv_calibration.ipynb`;
- keep the tracking configuration at `configs/botsort.yaml`.

Generated artefacts are written under `outputs/`.


## Imports


In [ ]:
import cv2
import csv
from ultralytics import YOLO
import ultralytics
import numpy as np
from scipy.optimize import linear_sum_assignment
import math
from pathlib import Path
from collections import defaultdict, Counter


## Optional model-training reference

The following cells document the training workflow used during project development. They remain disabled because training requires a separate labelled dataset, local credentials and substantial compute. They are not required to run inference with an existing `best.pt`.


In [ ]:
#Obter o dataset com imagens etiquetadas para ser possível treinar o modelo

# Optional training-data download (disabled in the repository).
# Never commit API keys. Configure credentials locally before using Roboflow.
#!pip install roboflow
#from roboflow import Roboflow

#rf = Roboflow(api_key="YOUR_LOCAL_API_KEY")
#project = rf.workspace("david-hrbak").project("detect-players-dgxz0-59epv")
#version = project.version(1)
#dataset = version.download("yolov8")
#Treinar o modelo com o ficheiro obtido acima. (Código demora bastante a correr)
#model = YOLO('yolov8n.pt')

#results = model.train(data="Detect-Players--1/data.yaml", epochs=100, patience=10, imgsz=1088)


## Geometric helper functions

Helper functions used to stabilise pitch-line references and homography-related anchoring. These functions reflect the experimental broadcast-video approach used in the original project.


In [ ]:
class ScalarPairEMA:
    def __init__(self, alpha=0.5):
        self.alpha = alpha
        self.v_top = None
        self.v_bot = None
    def update(self, vt, vb):
        if vt is None or vb is None:
            return self.v_top, self.v_bot
        if self.v_top is None or self.v_bot is None:
            self.v_top, self.v_bot = float(vt), float(vb)
        else:
            self.v_top = (1-self.alpha)*self.v_top + self.alpha*float(vt)
            self.v_bot = (1-self.alpha)*self.v_bot + self.alpha*float(vb)
        return self.v_top, self.v_bot

def warp_white_mask_rectified(frame_bgr, H, w, h):
    if H is None: return None
    wm = white_line_mask(frame_bgr)
    rect = cv2.warpPerspective(wm, H, (w, h))
    rect[: int(0.06*h), :] = 0  # remove HUD
    return rect

def warp_grass_mask_rectified(frame_bgr, H, w, h):
    if H is None: return None
    gm = grass_mask_bgr(frame_bgr)
    rect = cv2.warpPerspective(gm, H, (w, h))
    rect[: int(0.06*h), :] = 0
    return rect

def find_center_circle_u(rect_mask, w, h):
    if rect_mask is None: return None
    g = cv2.GaussianBlur(rect_mask, (9, 9), 2)
    circles = cv2.HoughCircles(g, cv2.HOUGH_GRADIENT, dp=1.2, minDist=h//3,
                               param1=120, param2=20,
                               minRadius=int(0.05*min(w,h)), maxRadius=int(0.20*min(w,h)))
    if circles is None: return None
    c = np.uint16(np.around(circles[0]))
    c = sorted(c, key=lambda x: (abs(int(x[0])-w//2), -int(x[2])))[0]
    return float(c[0])

def halfway_u_by_column_sum(rect_mask, w, h, u_prior=None):
    if rect_mask is None: return None
    x0, x1 = int(0.20*w), int(0.80*w)
    band = rect_mask[:, x0:x1]
    band = cv2.morphologyEx(band, cv2.MORPH_OPEN, np.ones((21,1), np.uint8), 1)
    col = band.astype(bool).sum(axis=0).astype(np.float32)
    if col.size == 0 or col.max() < 0.25*h: return None
    col = cv2.GaussianBlur(col.reshape(1,-1), (1,51), 0).ravel()
    u = x0 + int(np.argmax(col))
    if u_prior is not None and abs(u - u_prior) > 0.12*w:
        L, R = max(x0, int(u_prior - 0.10*w)), min(x1-1, int(u_prior + 0.10*w))
        if R > L:
            col2 = col[(L-x0):(R-x0+1)]
            if col2.size and col2.max() >= 0.20*h:
                u = L + int(np.argmax(col2))
    return float(u)

def touchlines_v_by_row_sum(rect_mask, w, h, x_margin=0.15):
    if rect_mask is None: return None, None
    x0, x1 = int(x_margin*w), int((1-x_margin)*w)
    band = rect_mask[:, x0:x1]
    band = cv2.morphologyEx(band, cv2.MORPH_OPEN, np.ones((1,21), np.uint8), 1)
    rows = band.astype(bool).sum(axis=1).astype(np.float32)
    if rows.size == 0: return None, None
    rows = cv2.GaussianBlur(rows.reshape(-1,1), (51,1), 0).ravel()
    top_win = rows[: int(0.40*h)]
    bot_win = rows[int(0.60*h):]
    if top_win.size == 0 or bot_win.size == 0: return None, None
    v_top = float(np.argmax(top_win))
    v_bot = float(int(0.60*h) + np.argmax(bot_win))
    req = 0.25 * (x1 - x0)
    if top_win[int(v_top)] < req or bot_win[int(v_bot - 0.60*h)] < req:
        return None, None
    return v_top, v_bot

def touchlines_v_by_grass_edges_adaptive(rect_grass, w, h, x_margin=0.12):
    if rect_grass is None: return None, None
    x0, x1 = int(x_margin*w), int((1-x_margin)*w)
    band = rect_grass[:, x0:x1] > 0
    rows = band.sum(axis=1).astype(np.float32)
    if rows.size == 0: return None, None
    rows = cv2.GaussianBlur(rows.reshape(-1,1), (51,1), 0).ravel()
    rows /= max(1.0, (x1 - x0))
    top_win = rows[: int(0.45*h)]
    bot_win = rows[int(0.55*h):]
    if top_win.size == 0 or bot_win.size == 0: return None, None
    t_thr = max(0.35, float(np.percentile(top_win, 60)))
    b_thr = max(0.25, float(np.percentile(bot_win, 50)))
    v_top = next((i for i,v in enumerate(top_win) if v >= t_thr), None)
    v_bot_rel = next((i for i,v in enumerate(bot_win[::-1]) if v >= b_thr), None)
    v_bot = (rows.size-1 - v_bot_rel) if v_bot_rel is not None else None
    if v_top is None and top_win.size: v_top = int(np.argmax(top_win))
    if v_bot is None and bot_win.size: v_bot = int(0.55*h) + int(np.argmax(bot_win))
    if v_top is None or v_bot is None or (v_bot - v_top) < 0.20*h: return None, None
    return float(v_top), float(v_bot)

def anchor_halfway_line(H, frame_bgr, w, h):
    if H is None: return None
    rect = warp_white_mask_rectified(frame_bgr, H, w, h)
    u_circ = find_center_circle_u(rect, w, h)
    u_line = halfway_u_by_column_sum(rect, w, h, u_prior=u_circ)
    if u_line is None: return H
    dx = 0.5*w - u_line
    Sx = np.array([[1.0, 0.0, dx],
                   [0.0, 1.0, 0.0],
                   [0.0, 0.0, 1.0]], dtype=float)
    return Sx @ H

def apply_y_anchor(H, v_top, v_bot, h):
    a = h / max(1.0, (v_bot - v_top)); b = -a * v_top
    Sy = np.array([[1.0, 0.0, 0.0],
                   [0.0, a,   b],
                   [0.0, 0.0, 1.0]], dtype=float)
    return Sy @ H

def anchor_touchlines_robust(H, frame_bgr, w, h, ema_pair):
    if H is None: return None
    rect_white = warp_white_mask_rectified(frame_bgr, H, w, h)
    vt, vb = touchlines_v_by_row_sum(rect_white, w, h)
    if vt is None or vb is None:
        rect_grass = warp_grass_mask_rectified(frame_bgr, H, w, h)
        vt2, vb2 = touchlines_v_by_grass_edges_adaptive(rect_grass, w, h)
        vt = vt if vt is not None else vt2
        vb = vb if vb is not None else vb2
    vt, vb = ema_pair.update(vt, vb)
    if vt is None or vb is None: return H
    return apply_y_anchor(H, vt, vb, h)

def draw_touchlines_overlay(frame_bgr, H, w, h, color=(0,255,0)):
    if H is None: return
    Hinv = np.linalg.inv(H)
    for vy in (0, h):
        pts = np.float32([[0, vy], [w, vy]]).reshape(-1,1,2)
        src = cv2.perspectiveTransform(pts, Hinv).reshape(-1,2)
        cv2.line(frame_bgr, tuple(map(int, src[0])), tuple(map(int, src[1])), color, 2, cv2.LINE_AA)


## Team-colour calibration

Replace these HSV ranges with values obtained for the specific match using `01_hsv_calibration.ipynb`. Manual HSV calibration was retained in the final project because it was more consistent than the tested clustering approach.


In [ ]:
#Inserir códigos HSV de cada equipa obtidos no script "Definir HSV" 
TEAM_A_HSV = { "h1": (0, 20),  "s": (120, 255), "v": (0, 255) }   # red
TEAM_B_HSV = { "h1": (0, 60),  "s": (0, 40),   "v": (50, 255) }   # white
REF_HSV    = None  # or e.g. {"h1": (20,35), "s": (80,255), "v": (80,255)}


## Detection, tracking, ID management and export

This section contains the main pipeline. Repository paths are resolved relative to the project root so the notebook can be executed from either the repository root or the `notebooks/` directory.


In [ ]:
# ========= REPOSITORY PATHS / USER SETTINGS =========
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = str(PROJECT_ROOT / "models" / "best.pt")
VIDEO_IN   = str(PROJECT_ROOT / "data" / "input" / "sample.mp4")
VIDEO_OUT  = str(OUTPUT_DIR / "tracking_video.mp4")
CSV_OUT    = str(OUTPUT_DIR / "tracking_output.csv")
TRACKER    = str(PROJECT_ROOT / "configs" / "botsort.yaml")

MIN_COVERAGE     = 0.08   # min fraction of jersey ROI must match a team color
SMOOTHING_VOTES  = 5      # per-track majority smoothing
DRAW_THICK       = 2

# Homography auto-detect/smoothing
H_UPDATE_EVERY   = 2      # frames between auto-estimates
H_SMOOTH_ALPHA   = 0.1    # EMA for H
# =================================

touchline_ema = ScalarPairEMA(alpha=0.4)
# ======== Pitch detection & homography utils ========
def detect_halfway_u(frame_bgr, H, w, h):
    """
    Returns the half-way line's u (in pixels in the rectified plane),
    or None if not found. Uses white-line mask + Hough, projects segments via H,
    keeps those that are near-vertical in rectified plane and near the center.
    """
    if H is None:
        return None
    wm = white_line_mask(frame_bgr)  # your existing function
    lines = cv2.HoughLinesP(wm, 1, np.pi/180, threshold=120,
                            minLineLength=min(w, h)//6, maxLineGap=8)
    if lines is None:
        return None

    ucands = []
    for x1, y1, x2, y2 in lines[:, 0]:
        pts = np.float32([[x1, y1], [x2, y2]]).reshape(-1, 1, 2)
        proj = cv2.perspectiveTransform(pts, H).reshape(-1, 2)  # pixels in rectified plane
        # vertical in rectified plane => small delta u, large delta v
        du = abs(proj[1, 0] - proj[0, 0])
        dv = abs(proj[1, 1] - proj[0, 1])
        if dv < 1:   # nearly horizontal in rectified plane; skip
            continue
        if du / dv > 0.25:  # not vertical enough
            continue
        u_mean = float(np.mean(proj[:, 0]))
        v_span = float(dv)
        # keep candidates that are somewhat central
        if 0.25 * w <= u_mean <= 0.75 * w:
            ucands.append((u_mean, v_span))

    if not ucands:
        return None
    # Prefer the longest vertical segment near the center
    ucands.sort(key=lambda t: (abs(t[0] - 0.5 * w), -t[1]))
    return ucands[0][0]


def anchor_halfway_line(H, frame_bgr, w, h):
    """
    Adjust H so the detected half-way line sits at u=0.5 (pixels: 0.5*w).
    Returns possibly-adjusted homography.
    """
    if H is None:
        return None
    u_half = detect_halfway_u(frame_bgr, H, w, h)
    if u_half is None:
        return H  # no change
    dx = 0.5 * w - u_half  # pixels in rectified plane
    S = np.array([[1.0, 0.0, dx],
                  [0.0, 1.0, 0.0],
                  [0.0, 0.0, 1.0]], dtype=float)
    return S @ H
    
def grass_mask_bgr(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    m1 = cv2.inRange(hsv, (35, 35, 40), (85, 255, 255))
    m2 = cv2.inRange(hsv, (30, 25, 40), (95, 255, 255))
    mask = cv2.bitwise_or(m1, m2)
    k = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, 1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, 2)
    return mask


def white_line_mask(img_bgr):
    # HSV: low saturation + high value
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    m_hsv = cv2.inRange(hsv, (0, 0, 170), (180, 70, 255))

    # LAB: high L (lights), robust under shade
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    L = lab[:, :, 0]
    m_lab = cv2.inRange(L, 200, 255)

    m = cv2.bitwise_or(m_hsv, m_lab)

    # Keep only on/near grass to remove ads/boards
    g = grass_mask_bgr(img_bgr)
    g = cv2.dilate(g, np.ones((9, 9), np.uint8), 2)
    m = cv2.bitwise_and(m, g)

    # Clean
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((3,3), np.uint8), 1)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8), 1)
    return m

def order_quad(pts):
    pts = np.array(pts, dtype=np.float32)
    s = pts.sum(axis=1)
    d = np.diff(pts, axis=1).ravel()
    tl = np.argmin(s); br = np.argmax(s)
    tr = np.argmin(d); bl = np.argmax(d)
    return np.array([pts[tl], pts[tr], pts[br], pts[bl]], dtype=np.float32)

def largest_quad_from_mask(mask, img_shape):
    h, w = img_shape[:2]
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    cnt = max(cnts, key=cv2.contourArea)
    if cv2.contourArea(cnt) < 0.1 * (h*w):
        return None
    rect = cv2.minAreaRect(cnt)
    box = cv2.boxPoints(rect)
    return order_quad(box)

def find_pitch_quad_auto(frame):
    gm = grass_mask_bgr(frame)
    quad = largest_quad_from_mask(gm, frame.shape)
    if quad is not None:
        return quad
    wm = white_line_mask(frame)
    cnts, _ = cv2.findContours(wm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    all_pts = np.vstack(cnts).reshape(-1,2)
    hull = cv2.convexHull(all_pts)
    if len(hull) < 4:
        return None
    rect = cv2.minAreaRect(hull)
    box = cv2.boxPoints(rect)
    return order_quad(box)

def homography_to_unit(quad, w, h):
    """Map pitch quad -> unit rectangle [0,w]x[0,h]; later we normalize by (w,h) to [0,1]."""
    dst = np.float32([[0,0],[w,0],[w,h],[0,h]])
    H, _ = cv2.findHomography(quad, dst, method=cv2.RANSAC, ransacReprojThreshold=3.0)
    return H

def feet_point_from_box(box):
    x1,y1,x2,y2 = box
    h = y2 - y1
    x = 0.5*(x1+x2)
    y = y2 + 0.06*h   # ~6% para baixo ajuda quando o corte come o pé
    return x, y

def project_feet_to_unit(boxes_xyxy, H, w, h):
    if H is None or len(boxes_xyxy) == 0:
        return [None]*len(boxes_xyxy)
    pts = []
    for x1,y1,x2,y2 in boxes_xyxy:
        x,y = feet_point_from_box((x1,y1,x2,y2))
        pts.append([x,y])
    pts = np.float32(pts).reshape(-1,1,2)
    proj = cv2.perspectiveTransform(pts, H).reshape(-1,2)
    proj[:,0] = np.clip(proj[:,0]/max(1,w), 0, 1)
    proj[:,1] = np.clip(proj[:,1]/max(1,h), 0, 1)
    return [tuple(p) for p in proj]

class HomographySmoother:
    def __init__(self, alpha=0.2):
        self.alpha = alpha
        self.H = None
    def update(self, H_new):
        if H_new is None:
            return self.H
        if self.H is None:
            self.H = H_new
            return self.H
        self.H = (1 - self.alpha) * self.H + self.alpha * H_new
        return self.H
# ====================================================


# ========= Team classify utils =========
def even_dims(img):
    h, w = img.shape[:2]
    eh, ew = h - (h % 2), w - (w % 2)
    return img if (h, w) == (eh, ew) else cv2.resize(img, (ew, eh), interpolation=cv2.INTER_AREA)

def hsv_mask_ratio(bgr, spec):
    if bgr.size == 0:
        return 0.0
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    h1l, h1h = spec["h1"]
    sL, sH = spec["s"]; vL, vH = spec["v"]
    mask = cv2.inRange(hsv, (h1l, sL, vL), (h1h, sH, vH))
    if spec.get("h2"):
        h2l, h2h = spec["h2"]
        mask |= cv2.inRange(hsv, (h2l, sL, vL), (h2h, sH, vH))
    return float((mask > 0).mean())

def jersey_roi(img, xyxy):
    # central 60% width, top 60% height (shirt region)
    x1, y1, x2, y2 = map(int, xyxy)
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(img.shape[1]-1, x2), min(img.shape[0]-1, y2)
    w, h = x2 - x1, y2 - y1
    if w <= 0 or h <= 0:
        return img[0:0, 0:0]
    cx1 = x1 + int(0.2 * w)
    cx2 = x2 - int(0.2 * w)
    cy1 = y1 + int(0.05 * h)
    cy2 = y1 + int(0.60 * h)
    return img[cy1:cy2, cx1:cx2]

def make_writer(path, fps, size):
    w, h = size
    fourcc = cv2.VideoWriter_fourcc(*'avc1')  # H.264 (QT-friendly if OpenCV+ffmpeg can encode)
    out = cv2.VideoWriter(path, fourcc, fps, (w, h))
    if out.isOpened():
        return out, path
    # Fallback to Motion-JPEG in .mov (also QuickTime-friendly)
    alt = path.rsplit('.', 1)[0] + ".mov"
    fourcc2 = cv2.VideoWriter_fourcc(*'MJPG')
    out2 = cv2.VideoWriter(alt, fourcc2, fps, (w, h))
    if not out2.isOpened():
        raise RuntimeError("Could not open any VideoWriter (H.264 or MJPEG).")
    return out2, alt

def draw_halfway_overlay(frame_bgr, H, w, h, color=(0,255,255)):
    if H is None:
        return
    Hinv = np.linalg.inv(H)
    # vertical line at u = 0.5*w in rectified plane
    pts = np.float32([[0.5*w, 0], [0.5*w, h]]).reshape(-1,1,2)
    src = cv2.perspectiveTransform(pts, Hinv).reshape(-1,2)
    p1 = tuple(map(int, src[0])); p2 = tuple(map(int, src[1]))
    cv2.line(frame_bgr, p1, p2, color, 2, cv2.LINE_AA)


# Colors (BGR)
COL_A   = (0, 0, 255)      # team A (red)
COL_B   = (255, 64, 0)     # team B
COL_REF = (0, 255, 255)
COL_UNK = (160, 160, 160)
# =====================================


# ======== Load model & classes ========
model = YOLO(MODEL_PATH)
names = model.names if hasattr(model, "names") else getattr(model.model, "names", {})
name_map = {int(k): str(v).lower() for k, v in (names.items() if isinstance(names, dict) else enumerate(names))}

ref_cls_ids = {k for k, v in name_map.items() if "referee" in v}
player_cls_ids = {k for k, v in name_map.items() if any(s in v for s in ("player", "person"))}
if not player_cls_ids:
    player_cls_ids = set(range(1000))


from collections import defaultdict
import math
import numpy as np

# =================== PARAMS ===================
IDM_MAX_PER_TEAM      = 10
IDM_LOCK_MIN_SCORE    = 0.08   # conf*cov mínimo para considerar forte quando a fechar plantel
IDM_REAPPEAR_MAX_GAP  = 300    # nº máx de frames para permitir reassociação de um slot desaparecido
IDM_DIST_THRESH_IMG   = 60.0   # px para considerar jogadores "muito próximos" (troca/cruzamento)
IDM_DIST_THRESH_FIELD = 0.1   # distância normalizada (u,v) para proximidade
IDM_SWAP_MARGIN       = 0.10   # melhoria relativa exigida para decidir swap de IDs

# ============== Helpers internos ==============

def _feet_from_box_xyxy(box):
    x1,y1,x2,y2 = box
    h = y2 - y1
    x = 0.5*(x1+x2)
    y = y2 + 0.06*h
    return float(x), float(y)


def _euclid(a, b):
    return float(np.hypot(a[0]-b[0], a[1]-b[1]))


class IDManager:
    """
    Gere IDs públicos estáveis com:
      - limite de 10/jogadores por equipa por frame (seleção por score)
      - fecho de plantéis (20 IDs fixos até ao fim)
      - reaparecimento reutiliza o mesmo ID (por proximidade)
      - correção de trocas em cruzamentos (swap usando cov/score)

    Espera por frame uma lista de detecções no formato:
        {
            'tid': tracker_id (int) ou None,
            'box': np.array([x1,y1,x2,y2], dtype=float),
            'feet': (img_x, img_y)  # em px
            'uv': (u,v) normalizado ou None,
            'conf': float,
            'covA': float,
            'covB': float,
            'team_pred': 'A'|'B'|'UNK'   # rótulo instantâneo antes do IDManager
        }
    E devolve uma lista de entradas geridas:
        {
            ...campos originais...,
            'public_id': int,
            'team': 'A'|'B',          # equipa final (pode corrigir o pred)
            'score': float            # score usado na seleção
        }
    """

    def __init__(self,
                 max_per_team=IDM_MAX_PER_TEAM,
                 lock_min_score=IDM_LOCK_MIN_SCORE,
                 reappear_gap=IDM_REAPPEAR_MAX_GAP,
                 swap_margin=IDM_SWAP_MARGIN,
                 dist_thresh_img=IDM_DIST_THRESH_IMG,
                 dist_thresh_field=IDM_DIST_THRESH_FIELD,
                debug=True):
        self.max_per_team = int(max_per_team)
        self.lock_min_score = float(lock_min_score)
        self.reappear_gap = int(reappear_gap)
        self.swap_margin = float(swap_margin)
        self.dist_thr_img = float(dist_thresh_img)
        self.dist_thr_field = float(dist_thresh_field)
        self.debug = debug

        self.next_pid = 1
        # roster por equipa: pid -> state
        self.roster = { 'A': {}, 'B': {} }
        # state: {
        #   'team': 'A'|'B', 'last_uv': (u,v)|None, 'last_feet': (x,y),
        #   'last_box': np.array, 'last_frame': int, 'strong_count': int
        # }
        self.roster_locked = False

    # -------------- scoring & labeling --------------
    def _scores(self, det):
        sA = float(det['conf'] * det['covA'])
        sB = float(det['conf'] * det['covB'])
        return sA, sB

    def _choose_team(self, det):
        sA, sB = self._scores(det)
        # Respeita cobertura mínima
        if det['covA'] < det.get('min_cov', 0.0) and det['covB'] < det.get('min_cov', 0.0):
            return 'UNK', 0.0
        if sA >= sB:
            return 'A', sA
        else:
            return 'B', sB

    # -------------- distances --------------
    def _dist_det_pid(self, det, pid_state):
        # Usa (u,v) se ambos tiverem; senão usa pés em px
        if det['uv'] is not None and pid_state.get('last_uv') is not None:
            duv = _euclid(det['uv'], pid_state['last_uv'])
            return duv, 'field'
        # fallback imagem
        dimg = _euclid(det['feet'], pid_state['last_feet']) if pid_state.get('last_feet') else 1e9
        return dimg, 'img'

    def _close_enough(self, det_i, det_j):
        # proximidade para considerar cruzamento
        if det_i.get('uv') is not None and det_j.get('uv') is not None:
            return _euclid(det_i['uv'], det_j['uv']) <= (2.0*self.dist_thr_field)
        return _euclid(det_i['feet'], det_j['feet']) <= self.dist_thr_img

    # -------------- assign helpers --------------
    def _assign_new_pid(self, team, det):
        pid = self.next_pid; self.next_pid += 1
        self.roster[team][pid] = {
            'team': team,
            'last_uv': det['uv'],
            'last_feet': det['feet'],
            'last_box': det['box'],
            'last_frame': det['frame_idx'],
            'strong_count': 1 if det['score'] >= self.lock_min_score else 0,
        }
        return pid

    def _update_pid(self, team, pid, det):
        st = self.roster[team][pid]
        st['last_uv'] = det['uv'] if det['uv'] is not None else st.get('last_uv')
        st['last_feet'] = det['feet']
        st['last_box'] = det['box']
        st['last_frame'] = det['frame_idx']
        if det['score'] >= self.lock_min_score:
            st['strong_count'] = st.get('strong_count', 0) + 1

    def _best_match_existing(self, team, det, candidate_pids):
        # devolve (pid, dist, space) ou (None, inf, '-')
        best_pid, best_d, best_space = None, 1e18, '-'
        for pid in candidate_pids:
            st = self.roster[team][pid]
            # janela de reaparecimento
            if det['frame_idx'] - st['last_frame'] > self.reappear_gap:
                continue
            d, space = self._dist_det_pid(det, st)
            if d < best_d:
                best_pid, best_d, best_space = pid, d, space
        return best_pid, best_d, best_space

    def _threshold_for_space(self, space):
        return self.dist_thr_field if space == 'field' else self.dist_thr_img

    def _can_lock(self):
        if self.roster_locked:
            return True
        A_ok = (len(self.roster['A']) >= self.max_per_team and
                all(self.roster['A'][pid].get('strong_count',0) >= 1 for pid in self.roster['A']))
        B_ok = (len(self.roster['B']) >= self.max_per_team and
                all(self.roster['B'][pid].get('strong_count',0) >= 1 for pid in self.roster['B']))
        if A_ok and B_ok:
            self.roster_locked = True
        return self.roster_locked

    # -------------- main step --------------
    def step(self, frame_idx, detections, min_cov=0.0):
        """detections: lista de dicts (ver docstring). Retorna lista atribuída."""
        # prepara dets com team/score
        dets = []
        for d in detections:
            d = dict(d)
            d['frame_idx'] = frame_idx
            d['min_cov'] = min_cov
            team, score = self._choose_team(d)
            if team == 'UNK':
                continue
            d['team'] = team
            d['score'] = score
            dets.append(d)

        # Seleção top-10 por equipa por score
        A = sorted([d for d in dets if d['team']=='A'], key=lambda z: z['score'], reverse=True)[:self.max_per_team]
        B = sorted([d for d in dets if d['team']=='B'], key=lambda z: z['score'], reverse=True)[:self.max_per_team]

        out = []

        # ---------- Atribuição quando ainda não está fechado ----------
        if not self.roster_locked:
            for team, L in (('A',A), ('B',B)):
                for det in L:
                    # tenta casar com pids existentes por proximidade
                    candidate_pids = list(self.roster[team].keys())
                    pid, d, space = self._best_match_existing(team, det, candidate_pids)
                    thr = self._threshold_for_space(space)
                    if pid is not None and d <= thr:
                        self._update_pid(team, pid, det)
                        det['public_id'] = pid
                    else:
                        # se ainda cabem ids nesta equipa, cria novo
                        if len(self.roster[team]) < self.max_per_team:
                            det['public_id'] = self._assign_new_pid(team, det)
                        else:
                            # equipa já lotada mas plantel ainda não fechado: escolhe o pid mais distante e substitui só se score muito maior
                            # (estratégia conservadora: normalmente NÃO criar/substituir)
                            det['public_id'] = -1  # descarta
                    if det['public_id'] != -1:
                        out.append(det)

            # verifica se já pode fechar plantéis
            self._can_lock()

        else:
            # ---------- Plantéis fechados: apenas IDs existentes ----------
            for team, L in (('A',A), ('B',B)):
                # pids disponíveis desta equipa
                free_pids = set(self.roster[team].keys())
                assigned_pids = set()

                # greedy por proximidade
                for det in L:
                    pid, d, space = self._best_match_existing(team, det, free_pids)
                    thr = self._threshold_for_space(space)
                    if pid is not None and d <= thr:
                        self._update_pid(team, pid, det)
                        det['public_id'] = pid
                        out.append(det)
                        free_pids.discard(pid)
                        assigned_pids.add(pid)
                    else:
                        # não encontrou match: tenta o mais próximo mesmo que esteja "perdido", para reaparecimento
                        pid2, d2, space2 = self._best_match_existing(team, det, self.roster[team].keys())
                        thr2 = self._threshold_for_space(space2)
                        if pid2 is not None and d2 <= thr2:
                            self._update_pid(team, pid2, det)
                            det['public_id'] = pid2
                            out.append(det)
                            free_pids.discard(pid2)
                            assigned_pids.add(pid2)
                        else:
                            det['public_id'] = -1  # não pertence a nenhum pid do plantel

            # --- Correção de cruzamentos (swap) ---
            # procura pares A-B muito próximos e decide se compensa trocar IDs
            changed = True
            # faz no máx 1 iteração para evitar loops
            if out:
                pairs = []
                for i in range(len(out)):
                    for j in range(i+1, len(out)):
                        if out[i]['team'] == out[j]['team']:
                            continue
                        if self._close_enough(out[i], out[j]):
                            pairs.append((i,j))

                for i,j in pairs:
                    a, b = out[i], out[j]
                    # equipas por pid (estão fechadas)
                    pidA_team = 'A' if a['public_id'] in self.roster['A'] else 'B'
                    pidB_team = 'A' if b['public_id'] in self.roster['A'] else 'B'
                    # scores que apoiam o estado atual
                    s_curr = (a['covA']*a['conf'] if pidA_team=='A' else a['covB']*a['conf']) + \
                             (b['covA']*b['conf'] if pidB_team=='A' else b['covB']*b['conf'])
                    # scores se trocarmos os pids
                    s_swap = (a['covA']*a['conf'] if pidB_team=='A' else a['covB']*a['conf']) + \
                             (b['covA']*b['conf'] if pidA_team=='A' else b['covB']*b['conf'])
                    if s_swap > s_curr * (1.0 + self.swap_margin):
                        # efetua swap de public_id
                        a_pid, b_pid = a['public_id'], b['public_id']
                        a['public_id'], b['public_id'] = b_pid, a_pid
                        # atualizar estados (posições associadas a cada pid)
                        self._update_pid(pidA_team, b_pid if pidA_team=='A' else a_pid, a)
                        self._update_pid(pidB_team, a_pid if pidB_team=='B' else b_pid, b)

        return out


# ======== Video init ========
cap = cv2.VideoCapture(VIDEO_IN)
fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
cap.release()

# Probe first frame for size & create writer
probe = next(model.track(source=VIDEO_IN, tracker=TRACKER, stream=True, persist=True, imgsz=1280,     # ou 1440 se couber em VRAM
    conf=0.2,
    iou=0.7), None)
if probe is None:
    raise RuntimeError("Could not read from video/source.")
frame0 = even_dims(probe.orig_img.copy())
H_img, W_img = frame0.shape[:2]
writer, out_path = make_writer(VIDEO_OUT, fps, (W_img, H_img))

# CSV
csv_file = open(CSV_OUT, "w", newline="")
csvw = csv.writer(csv_file)
csvw.writerow(["frame", "time_s", "jogador", "equipa",
               "img_x", "img_y", "coord_x", "coord_y", "conf",
               "cov_teamA", "cov_teamB"])

# Tracking state
votes = defaultdict(lambda: Counter())
final_label = {}
H_smooth = HomographySmoother(alpha=H_SMOOTH_ALPHA)

# Re-open full stream
stream = model.track(source=VIDEO_IN, tracker=TRACKER, stream=True, persist=True, imgsz=1280,     # ou 1440 se couber em VRAM
    conf=0.2,
    iou=0.7)

# --- IDManager ---
id_manager = IDManager(
    max_per_team=10,           # ponto 1
    lock_min_score=0.08,       # “conf*cov” mínimo para contar como forte (pode-se ajustar)
    reappear_gap=300,          # frames para recuperar id após desaparecer (≈10s a 30fps)
    swap_margin=0.10,          # margem relativa para aceitar swap
    dist_thresh_img=60.0,      # proximidade em px para candidatar swap/reatribuição
    dist_thresh_field=0.1     # proximidade em coords (u,v) normalizados
)

def iou_xyxy(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    iw, ih = max(0, x2-x1), max(0, y2-y1)
    inter = iw*ih
    if inter <= 0: return 0.0
    area = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/area


frame_idx = 0
for res in stream:
    frame = even_dims(res.orig_img.copy())
    h, w = frame.shape[:2]

    # ---- Auto-homography update/smoothing ----
    if frame_idx % H_UPDATE_EVERY == 0 or H_smooth.H is None:
        quad = find_pitch_quad_auto(frame)
        H_inst = homography_to_unit(quad, w, h) if quad is not None else None
        if H_inst is not None:
            H_inst = anchor_halfway_line(H_inst, frame, w, h)                 # X center on halfway
            H_inst = anchor_touchlines_robust(H_inst, frame, w, h, touchline_ema)  # robust Y using grass edges
        H_use = H_smooth.update(H_inst)
    else:
        H_use = H_smooth.H

    # ---- No boxes? write & continue ----
    if res.boxes is None or len(res.boxes) == 0:
        writer.write(frame)
        frame_idx += 1
        continue

    xyxy = res.boxes.xyxy.cpu().numpy()
    cls  = res.boxes.cls.cpu().numpy().astype(int) if res.boxes.cls is not None else np.zeros(len(xyxy), int)
    tids = res.boxes.id.cpu().numpy().astype(int)  if res.boxes.id  is not None else np.arange(len(xyxy))
    conf = res.boxes.conf.cpu().numpy()            if res.boxes.conf is not None else np.ones(len(xyxy))
    t_s  = frame_idx / float(fps)

    # Pre-compute projected feet (normalized field coords)
    uv_norm = project_feet_to_unit(xyxy, H_use, w, h)
    
    candidates = []

    for box, c, tid, cf, uv in zip(xyxy, cls, tids, conf, uv_norm):
            
        # Skip referees by class
        if c in ref_cls_ids:
            # draw ref box (optional)
            x1,y1,x2,y2 = map(int, box)
            cv2.rectangle(frame, (x1,y1), (x2,y2), COL_REF, DRAW_THICK)
            cv2.putText(frame, f"REF #{tid}", (x1, max(0, y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, COL_REF, 1, cv2.LINE_AA)
            #final_label[tid] = "REF"
            continue

        # Process only person/player classes
        if c not in player_cls_ids:
            continue
        

        # Team classification via jersey ROI HSV coverage
        roi = jersey_roi(frame, box)
        covA = hsv_mask_ratio(roi, TEAM_A_HSV)
        covB = hsv_mask_ratio(roi, TEAM_B_HSV)
        
        if REF_HSV:
            covR = hsv_mask_ratio(roi, REF_HSV)
            if covR >= max(covA, covB) and covR >= MIN_COVERAGE:
                label_now = "REF"
            else:
                if max(covA, covB) < MIN_COVERAGE:
                    label_now = "UNK"
                else:
                    label_now = "A" if covA >= covB else "B"
        else:
            if max(covA, covB) < MIN_COVERAGE:
                label_now = "UNK"
            else:
                label_now = "A" if covA >= covB else "B"

        # Majority vote smoothing per track id
        votes[tid][label_now] += 1
        if sum(votes[tid].values()) > SMOOTHING_VOTES:
            least_lbl, _ = votes[tid].most_common()[-1]
            votes[tid][least_lbl] -= 1
            if votes[tid][least_lbl] <= 0:
                del votes[tid][least_lbl]
        team_pred = votes[tid].most_common(1)[0][0] if votes[tid] else label_now
        
        
        feet_xy = (0.5*(box[0]+box[2]), box[3])
        candidates.append({
            'tid': int(tid),
            'box': box.astype(float),
            'feet': (float(feet_xy[0]), float(feet_xy[1])),
            'uv': (float(uv[0]), float(uv[1])) if uv is not None else None,
            'conf': float(cf),
            'covA': float(covA),
            'covB': float(covB),
            'team_pred': team_pred,   # apenas informativo
        })


    managed = id_manager.step(
        frame_idx=frame_idx,
        detections=candidates,
        min_cov=MIN_COVERAGE
    )
    

    for det in managed:
        pid = det['public_id']
        team = det['team']   # 'A' ou 'B'
        box = det['box']
        cf  = det['conf']
        covA, covB = det['covA'], det['covB']
        uv = det['uv']

        x1,y1,x2,y2 = map(int, box)
        color = COL_A if team=='A' else COL_B
        cv2.rectangle(frame, (x1,y1), (x2,y2), color, DRAW_THICK)
        cv2.putText(frame, f"ID {pid} - {team} {cf:.2f}", (x1, max(0, y1-6)),
cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

        # CSV
        if team in ('A','B'):
            img_x = 0.5*(box[0] + box[2])
            img_y = box[3]
            if uv is not None:
                u, v = uv
                u = f"{u:.4f}"; v = f"{v:.4f}"
            else:
                u = v = ""
            csvw.writerow([
                frame_idx, f"{(frame_idx/float(fps)):.3f}", int(pid), team,
                f"{img_x:.2f}", f"{img_y:.2f}",
                u, v,
                f"{float(cf):.3f}", f"{covA:.3f}", f"{covB:.3f}"
            ])
    

    #draw_halfway_overlay(frame, H_use, w, h)
    #draw_touchlines_overlay(frame, H_use, w, h)
    
    writer.write(frame)
    frame_idx += 1

writer.release()
csv_file.close()
print(f"✅ Saved annotated video to: {out_path}")
print(f"✅ CSV written to: {CSV_OUT}")
print("ℹ️ Field coords are normalized to [0,1]. If some frames lack H, those rows have empty field_u/field_v.")


## Outputs

When execution completes successfully, the main artefacts are:

- `outputs/tracking_output.csv` — frame-level tracking dataset;
- `outputs/tracking_video.mp4` — annotated tracking video.

Use the CSV as input to `03_tactical_metrics.ipynb`.
